In [1]:
import sys
sys.path.append("..")            # 저장소 루트 (project 패키지)
sys.path.append("../scripts")    # eval_silver 변환 함수 재사용

In [2]:
from pathlib import Path

BENCH_ID = "AIHub_KsponSpeech_general_other"
SILVER   = "/data/ASR/BENCHMARK/SILVER/AIHub_KsponSpeech_Other/transcript.jsonl"
MODEL    = "openai/whisper-small"
DEVICE   = "cuda:2"              # 실행 직전 nvidia-smi 로 빈 GPU 확인 후 지정

OUT_DIR = Path(f"../BENCHMARK/results/whisper_small__{BENCH_ID}")

In [3]:
from eval_silver import convert_silver

conv = OUT_DIR / "_silver_converted" / f"{BENCH_ID}.jsonl"
n = convert_silver(Path(SILVER), conv, corpus_id=BENCH_ID)
print(f"{n} samples → {conv}")

3000 samples → ../BENCHMARK/results/whisper_small__AIHub_KsponSpeech_general_other/_silver_converted/AIHub_KsponSpeech_general_other.jsonl


In [4]:
from project.data.adapters.whisper import build_predict_fn

predict_fn = build_predict_fn(
    MODEL, backbone=MODEL,
    language="ko", task="transcribe",
    beam_size=5, batch_size=16, device=DEVICE,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [5]:
from project.evaluation import evaluate_on_benchmark_suite

results = evaluate_on_benchmark_suite(
    model_name=f"whisper_small__{BENCH_ID}",
    predict_fn=predict_fn,
    benchmark_paths={BENCH_ID: conv},
    out_dir=OUT_DIR,
    batch_size=16,
)
results[BENCH_ID]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

CerResult(cer=20.148669388096007, scer=20.144678592875, wer=44.42154842508096, samples=3000, per_sample_cer=[23.333333333333332, 5.555555555555555, 11.11111111111111, 11.11111111111111, 15.789473684210526, 6.8181818181818175, 25.806451612903224, 11.11111111111111, 9.090909090909092, 27.77777777777778, 6.41025641025641, 22.857142857142858, 12.068965517241379, 5.555555555555555, 7.6923076923076925, 21.428571428571427, 11.538461538461538, 16.666666666666664, 13.333333333333334, 2.380952380952381, 7.6923076923076925, 36.507936507936506, 50.0, 50.0, 23.333333333333332, 25.0, 7.446808510638298, 26.229508196721312, 8.695652173913043, 15.555555555555555, 13.88888888888889, 29.545454545454547, 23.076923076923077, 28.57142857142857, 0.0, 16.666666666666664, 30.0, 0.0, 18.181818181818183, 14.117647058823529, 19.047619047619047, 47.368421052631575, 22.727272727272727, 50.0, 7.142857142857142, 25.0, 0.0, 46.15384615384615, 19.736842105263158, 12.280701754385964, 29.411764705882355, 100.0, 3.8461538

In [6]:
print((OUT_DIR / "evaluation_report.txt").read_text(encoding="utf-8"))

📊 ASR Evaluation Report — whisper_small__AIHub_KsponSpeech_general_other
   Date: 2026-06-12T17:10:36

## 1. Benchmark Set Results (한국어 CER 표준)
--------------------------------------------------------------------------------
Benchmark                                                  CER (%)   sCER (%)    Samples
--------------------------------------------------------------------------------
AIHub_KsponSpeech_general_other                              20.15      20.14      3,000
--------------------------------------------------------------------------------
Weighted Average                                             20.15                 3,000

## 2. Slice Analysis (메타 필드별)
--------------------------------------------------------------------------------

### AIHub_KsponSpeech_general_other
  [by age_group]
  value                   CER (%)    samples
  unknown                   20.15      3,000
  [by gender]
  value                   CER (%)    samples
  unknown                   20.

In [7]:
import json
import pandas as pd
import jiwer

lines = (OUT_DIR / BENCH_ID / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame(json.loads(l) for l in lines if l)

df["cer"] = [
    jiwer.cer(r, h) * 100 if r else float("nan")
    for r, h in zip(df["text_normalized"], df["prediction_normalized"])
]

worst = df.sort_values("cer", ascending=False).head(20)
for _, row in worst.iterrows():
    print(f"[CER {row.cer:5.1f}] 정답: {row.text_normalized}")
    print(f"             예측: {row.prediction_normalized}\n")

[CER 6600.0] 정답: 이피엘
             예측: 이피에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에

[CER 5970.0] 정답: 그치 으 서른이지.
             예측: 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다 감사합니다

[CER 3980.0] 정답: 어뜩하지.
             예측: 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 자 